In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_batch = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/seo_orders/seo_orders.csv")
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 7, Finished, Available, Finished, False)

In [6]:
df_watermark = (
    spark.read
    .format("delta")
    .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark")
)

orders_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_SEO_Orders")
    .select("last_watermark")
    .first()["last_watermark"]
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 8, Finished, Available, Finished, False)

In [7]:
df_incremental = df_batch.filter(
    F.col("updated_timestamp") > F.lit(orders_watermark)
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 9, Finished, Available, Finished, False)

In [8]:
window_orders = Window.partitionBy(
    "order_id"
).orderBy(
    F.col("updated_timestamp").desc()
)

df_clean = (
    df_incremental
    .withColumn("_rn", F.row_number().over(window_orders))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 10, Finished, Available, Finished, False)

In [9]:
df_clean = df_clean.withColumn(
    "country_code",
    F.when(
        F.upper(F.trim(F.col("country_code"))).isin(
            "USA", "UNITED STATES", "US"
        ),
        F.lit("US")
    )
    .otherwise(F.upper(F.trim(F.col("country_code"))))
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 11, Finished, Available, Finished, False)

In [10]:
df_clean = df_clean.filter(
    (F.upper(F.trim(F.col("order_status"))) == "COMPLETED") &
    (F.col("order_amount") > 0)
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 12, Finished, Available, Finished, False)

In [11]:
df_clean.createOrReplaceTempView("orders_incremental")

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 13, Finished, Available, Finished, False)

In [12]:
spark.sql("""
MERGE INTO fact_seo_orders AS target
USING orders_incremental AS source
ON target.order_id = source.order_id

WHEN MATCHED
     AND source.updated_timestamp > target.updated_timestamp
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *
""")

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 14, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [14]:
df_watermark = (
    spark.read
    .format("delta")
    .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark")
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 16, Finished, Available, Finished, False)

In [15]:
df_watermark_updated = (
    df_watermark
    .withColumn(
        "last_watermark",
        F.when(
            F.col("table_name") == "Fact_SEO_Traffic",
            F.lit("2026-05-31 23:27:00").cast("timestamp")
        )
        .when(
            F.col("table_name") == "Fact_Keyword_Performance",
            F.lit("2026-05-31 23:52:00").cast("timestamp")
        )
        .when(
            F.col("table_name") == "Fact_SEO_Orders",
            F.lit("2026-05-31 23:51:00").cast("timestamp")
        )
        .otherwise(F.col("last_watermark"))
    )
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 17, Finished, Available, Finished, False)

In [17]:
(
    df_watermark_updated.write
    .format("delta")
    .mode("overwrite")
    .save("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/control_watermark")
)

StatementMeta(, c161e6fb-5b11-4757-9414-bd90b8b55769, 19, Finished, Available, Finished, False)